# Zolai LLM Correction Pipeline & Human Review
This notebook runs the grammar correction pipeline on technical seed data using the Gemini API, and then provides a UI to review the results.

In [ ]:
!pip install -q google-generativeai pandas

import os
import json
import time
import urllib.request
import google.generativeai as genai
import pandas as pd
from IPython.display import display, HTML

try:
    urllib.request.urlopen('http://google.com')
    print("Internet connection: OK")
except:
    print("Internet connection: FAILED. Please enable Internet in Kaggle.")


In [ ]:
# === SETUP ===
# Add your Gemini API Key in Kaggle Secrets as 'GEMINI_API_KEY'
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    genai.configure(api_key=api_key)
    print("Gemini API configured successfully.")
except Exception as e:
    print(f"Warning: Could not load GEMINI_API_KEY from secrets: {e}")

DATA_DIR = "/kaggle/input/zolai-dataset-train/resources"
INPUT_FILE = os.path.join(DATA_DIR, "tech_seed_data_prompts.jsonl")
SYSTEM_PROMPT_FILE = os.path.join(DATA_DIR, "zolai_system_prompt.txt")
OUTPUT_DIR = "/kaggle/working/"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "tech_seed_data_fixed.jsonl")

In [ ]:
def load_system_prompt():
    if os.path.exists(SYSTEM_PROMPT_FILE):
        with open(SYSTEM_PROMPT_FILE, "r", encoding="utf-8") as f:
            return f.read()
    else:
        # Local fallback
        with open("../../resources/zolai_system_prompt.txt", "r", encoding="utf-8") as f:
            return f.read()

def process_with_gemini(model_name, system_prompt, user_prompt):
    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=system_prompt,
        generation_config={"response_mime_type": "application/json", "temperature": 0.2}
    )
    try:
        response = model.generate_content(user_prompt)
        return response.text
    except Exception as e:
        print(f"API Error: {e}")
        return None

In [ ]:
# Load input data
if not os.path.exists(INPUT_FILE):
    INPUT_FILE = "../../resources/tech_seed_data_prompts.jsonl"

items = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            items.append(json.loads(line))

print(f"Loaded {len(items)} prompts for correction.")

system_prompt = load_system_prompt()
MODEL = "gemini-1.5-pro"
LIMIT = 20 # Run on a small batch for testing/review

processed_count = 0
errors_count = 0

print(f"Starting correction using {MODEL} (Limit: {LIMIT})...")
with open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for data in items[:LIMIT]:
        prompt = data.get("llm_correction_prompt", "")
        if not prompt: continue

        result_str = process_with_gemini(MODEL, system_prompt, prompt)
        if result_str:
            try:
                correction_data = json.loads(result_str)
            except json.JSONDecodeError:
                clean_str = result_str.strip().strip('```json').strip('```').strip()
                try:
                    correction_data = json.loads(clean_str)
                except:
                    print("Failed to parse LLM JSON output.")
                    errors_count += 1
                    continue

            data["fixed_zolai"] = correction_data.get("corrected_text", "")
            data["back_translation"] = correction_data.get("back_translation", "")
            data["errors_found"] = correction_data.get("errors_found", [])
            
            if "llm_correction_prompt" in data:
                del data["llm_correction_prompt"]
            
            fout.write(json.dumps(data, ensure_ascii=False) + "\n")
            fout.flush()
            processed_count += 1
            print(f"[{processed_count}] Corrected: {data['fixed_zolai'][:50]}...")
            time.sleep(2) # Rate limit
        else:
            errors_count += 1
            time.sleep(5)

print(f"\nPipeline Complete! Processed: {processed_count}, Errors: {errors_count}")
print(f"Output saved to: {OUTPUT_FILE}")

## Human-in-the-Loop Review
Review the LLM corrections below. Compare the original text, the fixed Zolai text, and the back-translation to ensure grammatical accuracy.

In [ ]:
review_data = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                review_data.append(json.loads(line))

if review_data:
    # Convert list of errors into comma-separated strings for easy viewing
    for r in review_data:
        if isinstance(r.get('errors_found'), list):
            r['errors_found'] = ", ".join(r['errors_found'])
            
    df = pd.DataFrame(review_data)
    display_cols = ['text', 'fixed_zolai', 'back_translation', 'errors_found']
    df = df[[c for c in display_cols if c in df.columns]]
    
    # Use Pandas styling for a better review UI
    pd.set_option('display.max_colwidth', None)
    display(HTML(df.to_html(classes='table table-striped table-hover', escape=False)))
else:
    print("No corrected data found. Run the pipeline cell above first.")